# 15 - Word2Vec with Negative Sampling

Goal: Implement Word2Vec efficiently using negative sampling

Run with: conda activate tfenv

In [2]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import plotly.express as px
import pandas as pd
import time

print(f"TensorFlow version: {tf.__version__}")

I0000 00:00:1779771561.050802    9005 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TensorFlow version: 2.21.0


In [3]:
# Load WikiText-103 dataset
print("Loading iohadrubin/wikitext-103-raw-v1...")
from datasets import load_dataset

ds = load_dataset("iohadrubin/wikitext-103-raw-v1", split="train")
print(f"Dataset size: {len(ds)}")

Loading iohadrubin/wikitext-103-raw-v1...


/home/eanorambuena/miniconda/envs/tfenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset size: 29567


In [4]:
texts = [row['text'] for row in ds]
MAX_CHARS = 300_000
full_text = ''
for t in texts:
    full_text += t + ' '
    if len(full_text) > MAX_CHARS:
        break
print(f'Total chars: {len(full_text)}')

words = full_text.lower().split()
words = [w.strip('.,;:!?()[]\"\'-0123456789') for w in words]
words = [w for w in words if len(w) > 1]
print(f'Total words: {len(words)}')
print(f'Sample: {words[:20]}')

Total chars: 304654
Total words: 46763
Sample: ['valkyria', 'chronicles', 'iii', 'senjō', 'no', 'valkyria', 'unrecorded', 'chronicles', 'japanese', '戦場のヴァルキュリア', 'lit', 'valkyria', 'of', 'the', 'battlefield', 'commonly', 'referred', 'to', 'as', 'valkyria']


In [5]:
# Build vocabulary: top 10K words with count >= 5
from collections import Counter
word_counts = Counter(words)
vocab_common = word_counts.most_common(10000)
vocab = [w for w, c in vocab_common if c >= 2]
text_vocab = {word: idx for idx, word in enumerate(vocab)}
idx_to_word = {idx: word for word, idx in text_vocab.items()}
text_vocab_size = len(text_vocab)
text_embed_dim = 128
print(f"Vocabulary: {text_vocab_size} words")

Vocabulary: 3904 words


In [6]:
# Create training pairs (Skip-gram) — vectorizado
def create_pairs(words, window=2):
    word_ids = np.array([text_vocab.get(w, -1) for w in words], dtype=np.int32)
    valid = word_ids != -1
    valid_idx = np.where(valid)[0]
    targets, contexts = [], []
    for i in valid_idx:
        left = max(0, i - window)
        right = min(len(words), i + window + 1)
        ctx = np.concatenate([word_ids[left:i], word_ids[i+1:right]])
        ctx = ctx[ctx != -1]
        if len(ctx):
            targets.append(np.full(len(ctx), word_ids[i], dtype=np.int32))
            contexts.append(ctx)
    return np.concatenate(targets), np.concatenate(contexts)

train_words, train_context = create_pairs(words)
print(f"Training pairs: {len(train_words)}")

Training pairs: 156480


In [7]:
# Generate negative samples
# Negative sampling vectorizado
NEGATIVE_SAMPLING = 10
np.random.seed(42)

neg_indices = np.random.randint(0, text_vocab_size, size=(len(train_context), NEGATIVE_SAMPLING))
neg_words = np.repeat(train_context, NEGATIVE_SAMPLING)
neg_context = neg_indices.ravel()
neg_labels = np.zeros(len(neg_context), dtype=np.float32)

print(f"NEGATIVE_SAMPLING = {NEGATIVE_SAMPLING}")
print(f"Generated {len(neg_context)} negative samples")

NEGATIVE_SAMPLING = 10
Generated 1564800 negative samples


In [8]:
# Combine positive and negative samples
pos_labels = np.ones(len(train_words), dtype=np.float32)

all_words = np.concatenate([train_words, neg_words])
all_context = np.concatenate([train_context, neg_context])
all_labels = np.concatenate([pos_labels, neg_labels])

print(f"Total training samples: {len(all_words)}")
print(f"  Positive (1): {len(pos_labels)}")
print(f"  Negative (0): {len(neg_labels)}")

Total training samples: 1721280
  Positive (1): 156480
  Negative (0): 1564800


In [9]:
# Word2Vec with Negative Sampling - Custom Model
class Word2VecNS(keras.Model):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.target_embedding = layers.Embedding(
            vocab_size, embed_dim, name='embedding_target'
        )
        self.context_embedding = layers.Embedding(
            vocab_size, embed_dim, name='embedding_context'
        )
        # No Dense layer here, output is scalar logit
    def call(self, inputs):
        target, context = inputs
        target_emb = self.target_embedding(target)  # (batch, embed_dim)
        context_emb = self.context_embedding(context)  # (batch, embed_dim)
        dot_product = tf.reduce_sum(target_emb * context_emb, axis=-1, keepdims=True)  # (batch, 1)
        return tf.sigmoid(dot_product)  # (batch, 1)

model = Word2VecNS(text_vocab_size, text_embed_dim)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01),
    loss='binary_crossentropy',
    metrics=['accuracy']
    )

print("Word2Vec with Negative Sampling model ready!")

Word2Vec with Negative Sampling model ready!


W0000 00:00:1779771590.854184    9005 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [10]:
# Training with negative sampling
train_ds = tf.data.Dataset.from_tensor_slices(((all_words, all_context), all_labels))
train_ds = train_ds.shuffle(buffer_size=10000).batch(64).prefetch(tf.data.AUTOTUNE)

print("Training Word2Vec with Negative Sampling...")
start = time.time()

callbacks = [keras.callbacks.EarlyStopping(
    monitor='loss', patience=5, restore_best_weights=True
)]

history = model.fit(
    train_ds,
    epochs=20,
    verbose=1,
    callbacks=callbacks
)

print(f"Training time: {time.time() - start:.2f}s")

model.summary()

Training Word2Vec with Negative Sampling...
Epoch 1/20
26895/26895 ━━━━━━━━━━━━━━━━━━━━ 265s 10ms/step - accuracy: 0.9904 - loss: 0.0319
Epoch 2/20
26895/26895 ━━━━━━━━━━━━━━━━━━━━ 270s 10ms/step - accuracy: 0.9858 - loss: 0.1019
Epoch 3/20
26895/26895 ━━━━━━━━━━━━━━━━━━━━ 317s 12ms/step - accuracy: 0.9834 - loss: 0.1157
Epoch 4/20
26895/26895 ━━━━━━━━━━━━━━━━━━━━ 317s 12ms/step - accuracy: 0.9824 - loss: 0.1332
Epoch 5/20
26895/26895 ━━━━━━━━━━━━━━━━━━━━ 302s 11ms/step - accuracy: 0.9856 - loss: 0.1220
Epoch 6/20
26895/26895 ━━━━━━━━━━━━━━━━━━━━ 250s 9ms/step - accuracy: 0.9869 - loss: 0.1257
Training time: 1721.26s


Model: "word2_vec_ns"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_target (Embedding)    │ (None, 128)            │       499,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_context (Embedding)   │ (None, 128)            │       499,712 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,998,274 (11.44 MB)

 Trainable params: 999,424 (3.81 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 1,998,850 (7.63 MB)

In [11]:
# Get embeddings (use target embedding)
target_embeddings = model.target_embedding.get_weights()[0]
context_embeddings = model.context_embedding.get_weights()[0]

final_embeddings = (target_embeddings + context_embeddings) / 2

print(f"Target embeddings shape: {target_embeddings.shape}")
print(f"Context embeddings shape: {context_embeddings.shape}")

Target embeddings shape: (3904, 128)
Context embeddings shape: (3904, 128)


In [12]:
from numpy.linalg import norm

def cosine_similarity(a, b):
    return np.dot(a, b) / (norm(a) * norm(b) + 1e-8)

def find_similar(word, top_n=5):
    if word not in text_vocab:
        return []
    word_idx = text_vocab[word]
    word_emb = final_embeddings[word_idx]
    similarities = []
    for w, idx in text_vocab.items():
        if w != word:
            sim = cosine_similarity(word_emb, final_embeddings[idx])
            similarities.append((w, sim))
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[:top_n]

def analogy(word_a, word_b, word_c):
    if word_a not in text_vocab or word_b not in text_vocab or word_c not in text_vocab:
        return None
    vec = final_embeddings[text_vocab[word_b]] - final_embeddings[text_vocab[word_a]] + final_embeddings[text_vocab[word_c]]
    similarities = []
    for w, idx in text_vocab.items():
        if w not in [word_a, word_b, word_c]:
            sim = cosine_similarity(vec, final_embeddings[idx])
            similarities.append((w, sim))
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[0]

In [13]:
print("=== Similar Words (Negative Sampling) ===")
test_words = ['university', 'city', 'war', 'film', 'music', 'football']
for word in test_words:
    if word in text_vocab:
        similar = find_similar(word)
        print(f"'{word}' -> {[(w, f'{s:.3f}') for w, s in similar]}")

=== Similar Words (Negative Sampling) ===
'university' -> [('inches', '0.457'), ('september', '0.382'), ('floor', '0.372'), ('predecessor', '0.370'), ("couldn't", '0.368')]
'city' -> [('chapel', '0.547'), ('japanese', '0.540'), ('principal', '0.535'), ('waldrons', '0.533'), ('wales', '0.498')]
'war' -> [('met', '0.459'), ('mary', '0.457'), ('translated', '0.451'), ('sense', '0.450'), ('august', '0.450')]
'film' -> [('egyptians', '0.464'), ('projects', '0.448'), ('demons', '0.448'), ('accounts', '0.444'), ('akhenaten', '0.436')]
'music' -> [('rock', '0.472'), ('changing', '0.468'), ('tower', '0.458'), ('none', '0.453'), ('planning', '0.445')]
'football' -> [('ozawa', '0.410'), ('mustered', '0.406'), ('faulkner', '0.398'), ('psn', '0.389'), ('regiments', '0.386')]


In [14]:
print("=== Analogies ===")
result = analogy('king', 'man', 'queen')
if result:
    print(f"'king' - 'man' + 'queen' = '{result[0]}' ({result[1]:.3f})")

result = analogy('london', 'england', 'paris')
if result:
    print(f"'london' - 'england' + 'paris' = '{result[0]}' ({result[1]:.3f})")

=== Analogies ===
'king' - 'man' + 'queen' = 'elves' (0.552)
'london' - 'england' + 'paris' = 'argued' (0.339)


In [15]:
# Visualize in 3D with PCA
from sklearn.decomposition import PCA

top_words = [w for w, c in word_counts.most_common(200) if w in text_vocab]
top_indices = [text_vocab[w] for w in top_words]
top_embeddings = final_embeddings[top_indices]

pca = PCA(n_components=3)
embeddings_3d = pca.fit_transform(top_embeddings)

df = pd.DataFrame(embeddings_3d, columns=['x', 'y', 'z'])
df['word'] = top_words

fig = px.scatter_3d(df, x='x', y='y', z='z', text='word',
                 title='Word2Vec with Negative Sampling - Embeddings (PCA 3D)')
fig.update_traces(marker=dict(size=6), textposition='top center')
fig.update_layout(height=600, width=800)
fig.show()

In [16]:
# Summary
print(f"""
=== Summary ===
Vocab size: {text_vocab_size}
Embedding dim: {text_embed_dim}
Positive pairs: {len(train_words)}
Negative samples: {len(neg_labels)}
Total training: {len(all_words)}
Negative sampling ratio: {NEGATIVE_SAMPLING}:1

Model uses:
  - Two embeddings: target and context
  - Dot product + sigmoid (binary classification)
  - Much faster than full softmax!
""")



=== Summary ===
Vocab size: 3904
Embedding dim: 128
Positive pairs: 156480
Negative samples: 1564800
Total training: 1721280
Negative sampling ratio: 10:1

Model uses:
  - Two embeddings: target and context
  - Dot product + sigmoid (binary classification)
  - Much faster than full softmax!



In [17]:
# Guardar embeddings entrenados para uso futuro
np.save('target_embeddings.npy', model.target_embedding.get_weights()[0])
np.save('context_embeddings.npy', model.context_embedding.get_weights()[0])
np.save('text_vocab.npy', text_vocab)
print('Embeddings guardados como target_embeddings.npy y context_embeddings.npy')
print('Vocabulario guardado como text_vocab.npy')

Embeddings guardados como target_embeddings.npy y context_embeddings.npy
Vocabulario guardado como text_vocab.npy
